# LWS Visits - Angle Regression
We examine the relationship between targets' rotation angle and the probability that a visits is considered **Lookwing without Seeing** (LWS). We fit a logistic regression model predicting the probability of LWS visits based on the target's rotation angle, with random intercepts for each subject.

In [1]:
import time
from itertools import product

import numpy as np
import bambi as bmb
import arviz as az
import pandas as pd
import polars as pl
# from pymer4.models import glmer
import matplotlib.pyplot as plt
from scipy.special import expit
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import plotly.io as pio

import config as cnfg
from data_models.LWSEnums import SearchArrayCategoryEnum, ImageCategoryEnum

pio.renderers.default = "notebook"      # or "browser"

In [2]:
VERBOSE = True
EVENT_TYPE, FUNNEL_TYPE = "visit", "lws"
INITIAL_STEP = "instance_on_target"
SEED = 42

if FUNNEL_TYPE == "lws":
    STEPS = cnfg.LWS_FUNNEL_STEPS
elif FUNNEL_TYPE == "target_return":
    STEPS = cnfg.TARGET_RETURN_FUNNEL_STEPS
else:
    raise NotImplementedError(f"Unknown FUNNEL_TYPE: {FUNNEL_TYPE}")

## Prepare Data

In [3]:
from funnel__TODO_DELETE_THIS.prepare import prepare_funnel

DATA = prepare_funnel(
    data_dir=cnfg.OUTPUT_PATH,
    funnel_type=FUNNEL_TYPE,
    event_type=EVENT_TYPE,
    verbose=VERBOSE,
)
DATA = DATA.assign(abs_angle=lambda df: df["target_angle"].abs())
SUBSET = DATA[DATA[INITIAL_STEP]]   # Only consider events that passed the initial step

print(f"Empirical probability that a {EVENT_TYPE} is LWS given it passed the initial step `{INITIAL_STEP}`:\t{100 * DATA['final'].sum() / DATA[INITIAL_STEP].sum() :.2f}%")

Calculating Funnel Steps:  20%|██        | 2/10 [00:00<00:00, 18.82it/s]


ValueError: Data must be 1-dimensional, got ndarray of shape (720, 14) instead

## Simple Hierarchical Model
We fit a hierarchical logistic regression model predicting the probability of an event passing the `final` funnel step given it passed the `INITIAL_STEP`.<br>
In the **simple model** we only include the (absolute) target rotation-angle as a predictor, with random intercepts for each subject:
$$logit(final) \sim 1 + abs(target\_angle) + (1 | subject) $$

In [ ]:
FORMULA_1 = "final ~ 1 + abs_angle + (1 | subject)"

### (1) Frequentist Model

In [ ]:
freq_model1 = glmer(formula=FORMULA_1, data=pl.from_pandas(SUBSET), family="binomial",)
freq_model1.set_factors({
    "subject": SUBSET["subject"].unique().sort_values().map(str, na_action="").tolist(),
})
results = freq_model1.fit(summary=True, exponentiate=False)
results

In [ ]:
freq_model1.emmeans("abs_angle")

### (2) Bayesian Model

In [ ]:
start = time.time()

bayes_model1 = bmb.Model(FORMULA_1, SUBSET, family="bernoulli")
bayes_idata1 = bayes_model1.fit(
    draws=2000, tune=1000, chains=4, cores=2, target_accept=0.95, progressbar=False, random_seed=SEED,
)

elapsed = time.time() - start
print(f"Model fitting completed in {int(elapsed // 3600)}:{int((elapsed % 3600) // 60)}:{elapsed % 60:.2f} (hh:mm:ss)")

In [ ]:
az.summary(bayes_idata1, var_names=["Intercept", "abs_angle"], round_to=4)

In [ ]:
az.plot_trace(bayes_idata1, var_names=["Intercept", "abs_angle"])

## Hierarchical Multi-Regression - Trial Category
We extend the original model by including the trial category as an additional predictor, as well as the interaction between trial category and target rotation angle. Like before, we include random intercepts for each subject:
$$logit(final) \sim 1 + abs(target\_angle) * C(trial\_category) + (1 | subject) $$

In [ ]:
FORMULA_2 = "final ~ 1 + abs_angle * trial_category + (1 | subject)"

### (1) Frequentist Model

In [ ]:
freq_model2 = glmer(formula=FORMULA_2, data=pl.from_pandas(SUBSET), family="binomial",)
freq_model2.set_factors({
    "subject": SUBSET["subject"].unique().sort_values().map(str, na_action="").tolist(),
    "trial_category": SUBSET["trial_category"].unique().sort_values().tolist(),
})
results = freq_model2.fit(summary=True, exponentiate=False)
results

In [ ]:
freq_model2.emmeans("abs_angle")

### (2) Bayesian Model

In [ ]:
start = time.time()

bayes_model2 = bmb.Model(FORMULA_2, SUBSET, family="bernoulli")
bayes_idata2 = bayes_model2.fit(
    draws=2000, tune=1000, chains=4, cores=2, target_accept=0.95, progressbar=False, random_seed=SEED,
)

elapsed = time.time() - start
print(f"Model fitting completed in {int(elapsed // 3600)}:{int((elapsed % 3600) // 60)}:{elapsed % 60:.2f} (hh:mm:ss)")

In [ ]:
az.summary(bayes_idata2, var_names=["Intercept", "abs_angle", "trial_category", "abs_angle:trial_category"], round_to=4)

In [ ]:
az.plot_trace(bayes_idata2, var_names=["Intercept", "abs_angle", "trial_category"])

## Hierarchical Multi-Regression - Target Category
We extend the original model by including the target category as an additional predictor, as well as the interaction between target category and target rotation angle. Like before, we include random intercepts for each subject:
$$logit(final) \sim 1 + abs(target\_angle) * C(target\_category) + (1 | subject) $$

In [ ]:
FORMULA_3 = "final ~ 1 + abs_angle * target_category + (1 | subject)"

### (1) Frequentist Model

In [ ]:
freq_model3 = glmer(formula=FORMULA_3, data=pl.from_pandas(SUBSET), family="binomial",)
freq_model3.set_factors({
    "subject": SUBSET["subject"].unique().sort_values().map(str, na_action="").tolist(),
    "target_category": SUBSET["target_category"].unique().sort_values().tolist()
})
results = freq_model3.fit(summary=True, exponentiate=False)
results

In [ ]:
freq_model3.emmeans("abs_angle")

### (2) Bayesian Model

In [ ]:
start = time.time()

bayes_model3 = bmb.Model(FORMULA_3, SUBSET, family="bernoulli")
bayes_idata3 = bayes_model3.fit(
    draws=2000, tune=1000, chains=4, cores=2, target_accept=0.95, progressbar=False, random_seed=SEED,
)

elapsed = time.time() - start
print(f"Model fitting completed in {int(elapsed // 3600)}:{int((elapsed % 3600) // 60)}:{elapsed % 60:.2f} (hh:mm:ss)")

In [ ]:
az.summary(
    bayes_idata3,
    var_names=["Intercept", "abs_angle", "target_category", "abs_angle:target_category"],
    round_to=4
)

In [ ]:
az.plot_trace(bayes_idata3, var_names=["Intercept", "abs_angle", "target_category"])

## Complete Hierarchical Regression
We apply the complete model, examining the effects of `target_angle`, `target_category`, `trial_category` and their interactions on the probability of LWS. Like before, we include random intercepts for each subject:
$$logit(final) \sim 1 + abs(target\_angle) * C(target\_category) * C(trial\_category) + (1 | subject) $$

In [ ]:
FORMULA_4 = "final ~ 1 + abs_angle * target_category * trial_category + (1 | subject)"

### (1) Frequentist Model

In [ ]:
freq_model4 = glmer(formula=FORMULA_4, data=pl.from_pandas(SUBSET), family="binomial",)
freq_model4.set_factors({
    "subject": SUBSET["subject"].unique().sort_values().map(str, na_action="").tolist(),
    "target_category": SUBSET["target_category"].unique().sort_values().tolist(),
    "trial_category": SUBSET["trial_category"].unique().sort_values().tolist(),
})
results = freq_model4.fit(summary=True, exponentiate=False)
results

In [ ]:
freq_model4.emmeans("abs_angle")

In [ ]:
freq_model4.emmeans("trial_category", contrasts="pairwise", p_adjust="tukey")

In [ ]:
freq_model4.emmeans(
    "target_category",
    contrasts={
        "HUMAN_minus_ANIMAL": [-0.5, 0.5, -0.5, 0.5, 0, 0],
        "FACE_minus_OTHER": [-0.5, -0.5, 0.5, 0.5, 0, 0],
        "ANIMATE_minus_INANIMATE": [0.25, 0.25, 0.25, 0.25, -0.5, -0.5]
    },
    p_adjust="sidak",
    type="response",
)

### (2) Bayesian Model

In [ ]:
start = time.time()

bayes_model4 = bmb.Model(FORMULA_4, SUBSET, family="bernoulli")
bayes_idata4 = bayes_model4.fit(
    draws=2000, tune=1000, chains=4, cores=2, target_accept=0.95, progressbar=False, random_seed=SEED,
)

elapsed = time.time() - start
print(f"Model fitting completed in {int(elapsed // 3600)}:{int((elapsed % 3600) // 60)}:{elapsed % 60:.2f} (hh:mm:ss)")

In [ ]:
az.summary(
    bayes_idata4,
    var_names=["Intercept", "abs_angle", "target_category", "trial_category"],
    round_to=4
)

In [ ]:
az.plot_trace(bayes_idata4, var_names=["Intercept", "abs_angle", "target_category", "trial_category"])